# Set up

## Load libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
from matplotlib import pyplot as plt

# Check
## Get run date
import datetime
## Get URIs
import awswrangler as wr
## Clean text
import string

## Declare constants

In [2]:
# Check
## Get data URIs
data_uri_prefix_sr = 's3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs'
## Get preprocessor URIs
preprocessor_uri_prefix_sr = 's3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor'

# Check

## Get run date

In [3]:
str(datetime.datetime.now())

'2023-10-13 00:35:51.642120'

## Get data URIs

In [4]:
data_uris_ss = pd.Series(data=wr.s3.list_objects(path=data_uri_prefix_sr))

print(*data_uris_ss, sep='\n')

s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_test_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_test_noleaks_raw_1.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_train_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_train_noleaks_raw_1.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_valid_noleaks.gzip
s3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_valid_noleaks_raw_1.gzip


## Get URI

In [5]:
data_uri_sr = (
    data_uris_ss
    .loc[data_uris_ss.str.contains(pat='train') & data_uris_ss.str.contains(pat='raw')]
    .squeeze())

data_uri_sr

's3://20231010-gen-xii/01_ad/01_data_prep/05_leaky_features/04_write_dfs/df_train_noleaks_raw_1.gzip'

## Read in data

In [6]:
X_train = pd.read_parquet(path=data_uri_sr)

X_train.info()
X_train

<class 'pandas.core.frame.DataFrame'>
Index: 12604 entries, 613648 to 345224
Columns: 1566 entries, inttype__app to data_set
dtypes: datetime64[us](1), float64(1531), int64(1), object(33)
memory usage: 150.7+ MB


,inttype__app,linkf060__tu,linkf045__tu,linkf079__tu,linkf185__tu,linkf195__tu,linkf193__tu,linkf105__tu,linkb012__tu,linkf032__tu,...,fltdowncash__app,fltallowance__app,bitdealerapplicantsamezip__app,bitdealerapplicantsamecity__app,bitdealerapplicantsamestate__app,uniqueid,applicationdate__app,target,year_month,data_set
613648,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,0.0,0.0,0.0,0.0,0.0,2.284354e+14,2015-10-21,0,2015-10,train
54406,NaN,NaN,NaN,1071.0,470.0,1088.0,470.0,374.0,0.0,N,...,0.0,0.0,0.0,0.0,1.0,1.423462e+14,2013-12-20,1,2013-12,train
1241911,NaN,0.0,0.0,NaN,225.0,2480.0,225.0,225.0,NaN,N,...,0.0,0.0,0.0,0.0,1.0,3.346873e+14,2017-09-23,1,2017-09,train
738908,NaN,0.0,0.0,0.0,73.0,73.0,73.0,73.0,NaN,N,...,0.0,0.0,0.0,0.0,1.0,2.476830e+14,2016-03-05,0,2016-03,train
567065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,...,0.0,0.0,0.0,0.0,1.0,2.211479e+14,2015-08-28,0,2015-08,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
780225,NaN,NaN,NaN,20.0,330.0,1944.0,330.0,330.0,0.0,N,...,0.0,0.0,0.0,1.0,1.0,2.540843e+14,2016-04-18,0,2016-04,train
1036445,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,0.0,0.0,0.0,1.0,1.0,2.956538e+14,2017-01-20,0,2017-01,train
13993,7.0,0.0,0.0,0.0,1034.0,1070.0,1034.0,1034.0,NaN,N,...,0.0,0.0,0.0,0.0,1.0,1.359528e+14,2013-10-21,0,2013-10,train
359605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,0.0,0.0,0.0,0.0,1.0,1.895910e+14,2015-01-01,0,2015-01,train


## Split into target vector and feature matrix

In [7]:
print(f'Shape: {X_train.shape}')

target_sr = 'target'
y_train = X_train.pop(item=target_sr)

print(f'Shape: {X_train.shape}')

Shape: (12604, 1566)
Shape: (12604, 1565)


## Get preprocessor URIs

In [8]:
preprocessor_uris_ss = pd.Series(data=wr.s3.list_objects(path=preprocessor_uri_prefix_sr))

print(*preprocessor_uris_ss, sep='\n')

s3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/cls_model_preprocessing.pkl
s3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/df_dollars.csv
s3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/preprocessing.py


## Copy

In [9]:
for preprocessor_uri_sr in preprocessor_uris_ss:
    !aws s3 cp $preprocessor_uri_sr .

download: s3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/cls_model_preprocessing.pkl to ./cls_model_preprocessing.pkl
download: s3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/df_dollars.csv to ./df_dollars.csv
download: s3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/preprocessing.py to ./preprocessing.py


## Load module

In [10]:
import preprocessing

## Get URI

In [11]:
preprocessor_uri_sr = (
    preprocessor_uris_ss
    .loc[preprocessor_uris_ss.str.contains(pat='pkl')]
    .squeeze())

preprocessor_uri_sr

's3://20231010-gen-xii/01_ad/02_model/00_preprocessing/01_create_preprocessor/cls_model_preprocessing.pkl'

## Read in preprocessor

In [12]:
pm = pd.read_pickle(filepath_or_buffer=preprocessor_uri_sr)

pm

PreprocessingModel(list_transformers=[ReplaceNaNs(str_message='NaN Replacer'),
                                      ReplaceBooleans(str_message='Boolean '
                                                                  'Replacer'),
                                      SetDataTypes(),
                                      CleanText(list_cols=['city__tu',
                                                           'linkf001__tu',
                                                           'linkf032__tu',
                                                           'rvlr14__tu',
                                                           'rvlr15__tu',
                                                           'rvlr16__tu',
                                                           'rvlr17__tu',
                                                           'rvlr18__tu',
                                                           'rvlr19__tu',
                                                           'rvlr20__tu',
                                                           'rvlr21__tu',
                                                           'rvlr22__tu',
                                                           'rvlr23__tu',
                                                           'rvlr24__tu',
                                                           'rvl...
                                      ReplaceInf(str_message='Replace inf and '
                                                             '-inf with NaN'),
                                      Imputer(), MapTerm(), MapPTI(),
                                      RoundBinning(dict_round={'fltamountfinanced__app': 500,
                                                               'fltapproveddowntotal__app': 500,
                                                               'fltapprovedpricewholesale__app': 500,
                                                               'fltapprovedservicecontract__app': 500,
                                                               'fltdowncash__app': 500,
                                                               'fltgapinsurance__app': 500,
                                                               'intservicecontractmileage__app': 10000},
                                                   str_message='Round values')])

## Get steps

In [13]:
print(*map(lambda x: x.__class__.__name__, pm.list_transformers), sep='\n')

ReplaceNaNs
ReplaceBooleans
SetDataTypes
CleanText
Inflator
Inflator
ClipValues
ClipValues
CustomImputer
Imputer
FeatureValueReplacer
DateFeatures
RoundBinning
FeatureEngineering
ReplaceInf
Imputer
MapTerm
MapPTI
RoundBinning


## Test replace NaNs

### Does it do anything?

In [14]:
tmp = pm.list_transformers[0]
to_replace_lt = ['None', None, 'NaN', 'nan', '']

print(f'Before: {X_train.map(func=lambda x: x in to_replace_lt).sum().sum():,}')

X_train = tmp.transform(X=X_train)

print(f'After: {X_train.map(func=lambda x: x in to_replace_lt).sum().sum():,}')

Before: 15,669
NaN Replacer: 0.28598 sec.
After: 0


### Would it do anything?

In [15]:
size_te = (2**3, 2**2)
test_df = pd.DataFrame(data=np.random.choice(a=to_replace_lt, size=size_te))

print(f'Before: {test_df.map(func=lambda x: x in to_replace_lt).sum().sum():,}')
display(test_df)

test_df = tmp.transform(X=test_df)

print(f'After: {test_df.map(func=lambda x: x in to_replace_lt).sum().sum():,}')
display(test_df)

Before: 32


,0,1,2,3
0,None,,NaN,NaN
1,None,nan,None,None
2,None,nan,,nan
3,None,,NaN,None
4,,NaN,None,None
5,NaN,NaN,nan,NaN
6,,None,None,NaN
7,None,NaN,nan,None


NaN Replacer: 0.0021344 sec.
After: 0


,0,1,2,3
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN


## Test replace booleans

### Does it do anything?

In [16]:
tmp = pm.list_transformers[1]
to_replace_dt = {
    True: 1,
    False: 0,
    'True': 1,
    'False': 0,
    '0': 0,
    '1': 1}
to_replace_lt = list(to_replace_dt.keys())

print(f'Before: {X_train.map(func=lambda x: x in to_replace_lt).sum().sum():,}')

X_train = tmp.transform(X=X_train)

print(f'After: {X_train.map(func=lambda x: x in to_replace_lt).sum().sum():,}')

Before: 5,221,667
Boolean Replacer: 0.15733 sec.
After: 5,221,667


### Would it do anything?

In [17]:
test_df = pd.DataFrame(data=np.random.choice(a=to_replace_lt, size=size_te))

print(f'Before: {test_df.map(func=lambda x: x in to_replace_lt).sum().sum():,}')
display(test_df)

test_df = tmp.transform(X=test_df)

print(f'After: {test_df.map(func=lambda x: x in to_replace_lt).sum().sum():,}')
display(test_df)

Before: 32


,0,1,2,3
0,False,False,1,0
1,0,False,False,True
2,0,False,False,False
3,False,False,False,False
4,True,True,False,1
5,False,True,True,False
6,False,0,1,False
7,False,False,True,False


Boolean Replacer: 0.0017416 sec.
After: 32


,0,1,2,3
0,0,0,1,0
1,0,0,0,1
2,0,0,0,0
3,0,0,0,0
4,1,1,0,1
5,0,1,1,0
6,0,0,1,0
7,0,0,1,0


## Test set data types

### Fit

In [18]:
tmp = pm.list_transformers[2]

tmp.fit(X=X_train)

SetDataTypes()

### Does it do anything?

In [19]:
print(f'Before:', X_train.dtypes.value_counts(), sep='\n')

X_train = tmp.transform(X=X_train)

print(f'Before:', X_train.dtypes.value_counts(), sep='\n')

Before:
float64           1531
object              33
datetime64[us]       1
Name: count, dtype: int64


100%|██████████| 1565/1565 [00:00<00:00, 2696.74it/s]

Data Type Setter: 0.77833 sec.
Before:
float64           1531
object              33
datetime64[us]       1
Name: count, dtype: int64


### Would it do anything?

In [20]:
test_df = X_train.copy().astype(dtype=str)

print(f'Before:', test_df.dtypes.value_counts(), sep='\n')

test_df = tmp.transform(X=test_df)

print(f'Before:', test_df.dtypes.value_counts(), sep='\n')

Before:
object    1565
Name: count, dtype: int64


100%|██████████| 1565/1565 [00:04<00:00, 359.99it/s]

Data Type Setter: 4.5419 sec.
Before:
float64           1531
object              33
datetime64[us]       1
Name: count, dtype: int64


## Test clean text

### Does it do anything?

In [21]:
tmp = pm.list_transformers[3]
to_replace_lt = list(string.ascii_uppercase + ' ')

print(f'Before: {X_train.map(func=lambda x: x in to_replace_lt).sum().sum():,}')

X_train = tmp.transform(X=X_train)

print(f'After: {X_train.map(func=lambda x: x in to_replace_lt).sum().sum():,}')

Before: 41,464


100%|██████████| 32/32 [00:00<00:00, 69.29it/s]


Clean text and impute non-numeric: 0.47344 sec.
After: 0


### Would it do anything?

In [22]:
test_df = pd.DataFrame(
    data=np.random.choice(a=list('ABC') + [' A ', 'A B'], size=size_te),
    columns=tmp.list_cols[:size_te[1]])

print(f'Before: {test_df.map(func=lambda x: x in to_replace_lt).sum().sum():,}')
display(pd.concat(objs=[test_df, test_df.map(func=len)], axis=1))

test_df = tmp.transform(X=test_df)

print(f'After: {test_df.map(func=lambda x: x in to_replace_lt).sum().sum():,}')
display(pd.concat(objs=[test_df, test_df.map(func=len)], axis=1))

Before: 16


,city__tu,linkf001__tu,linkf032__tu,rvlr14__tu,city__tu,linkf001__tu,linkf032__tu,rvlr14__tu
0,C,A B,A,B,1,3,3,1
1,C,A,C,A,1,3,1,3
2,C,A B,A,C,1,3,3,1
3,A,C,A B,A B,1,1,3,3
4,A B,A B,A B,A,3,3,3,3
5,A B,B,C,B,3,1,1,1
6,B,A B,C,C,1,3,1,1
7,C,A,A B,C,1,3,3,1


100%|██████████| 4/4 [00:00<00:00, 713.47it/s]

Clean text and impute non-numeric: 0.0082742 sec.
After: 0


,city__tu,linkf001__tu,linkf032__tu,rvlr14__tu,city__tu,linkf001__tu,linkf032__tu,rvlr14__tu
0,c,ab,a,b,1,2,1,1
1,c,a,c,a,1,1,1,1
2,c,ab,a,c,1,2,1,1
3,a,c,ab,ab,1,1,2,2
4,ab,ab,ab,a,2,2,2,1
5,ab,b,c,b,2,1,1,1
6,b,ab,c,c,1,2,1,1
7,c,a,ab,c,1,1,2,1


## Test inflator

### Does it do anything?

In [23]:
tmp = pm.list_transformers[4]
columns_lt = X_train.columns.intersection(other=tmp.list_cols)

print('Before:')
display(X_train.groupby(by=X_train[tmp.str_datecol].dt.year)[columns_lt].max().T)

X_train = tmp.transform(X=X_train)

print(pd.Series(data=tmp.dict_inflation_rate))
print('After:')
display(X_train.groupby(by=X_train[tmp.str_datecol].dt.year)[columns_lt].max().T)

Before:


applicationdate__app,2013,2014,2015,2016,2017
fltservicecontract__app,3650.00,7063.00,4500.0,5079.00,3690.00
fltgapinsurance__app,895.00,900.00,1900.0,1090.00,995.00
fltapprovedpricewholesale__app,92277.00,40528.00,258985.0,157075.00,51745.00
aut232__tu,50147.00,38099.00,53191.0,67797.00,38327.00
aut233__tu,50147.00,55056.00,67317.0,105384.00,77864.00
aut234__tu,50147.00,91348.00,92084.0,131251.00,109927.00
aut235__tu,60669.00,103705.00,119989.0,152605.00,109953.00
aut231__tu,14690.00,38099.00,53191.0,17367.00,38327.00
au101s__tu,76723.00,96264.00,92892.0,116549.00,85240.00
au57s__tu,5564.00,17995.00,14609.0,17793.00,9225.00


/home/ec2-user/SageMaker/20231010_gen_xii/ad_hoc/check_preprocessing/preprocessing.py:169: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['year'] = X[self.str_datecol].dt.year
/home/ec2-user/SageMaker/20231010_gen_xii/ad_hoc/check_preprocessing/preprocessing.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['factor'] = X['year'].map(self.dict_inflation_rate)
100%|██████████| 15/15 [00:00<00:00, 3320.91it/s]

Inflate to 2021 dollars (automobiles only): 0.018109 sec.
2013    1.134349
2014    1.135868
2015    1.136342
2016    1.142522
2017    1.157902
2018    1.154958
2019    1.151207
dtype: float64
After:


applicationdate__app,2013,2014,2015,2016,2017
fltservicecontract__app,4140.374612,8022.639123,5113.538917,5802.869892,4272.657113
fltgapinsurance__app,1015.242542,1022.281638,2159.049765,1245.349120,1152.112148
fltapprovedpricewholesale__app,104674.341946,46034.478035,294295.528068,179461.663374,59915.621220
aut232__tu,56884.209777,43275.453480,60443.166336,77459.572763,44378.896792
aut233__tu,56884.209777,62536.375412,76495.133166,120403.552016,90158.854588
aut234__tu,56884.209777,103759.314536,104638.915021,149957.171921,127284.655404
aut235__tu,68819.832153,117795.241428,136348.538013,174354.589458,127314.760847
aut231__tu,16663.589878,43275.453480,60443.166336,19842.181810,44378.896792
au101s__tu,87030.674352,109343.244018,105557.079342,133159.811584,98699.537207
au57s__tu,6311.518998,20439.953421,16600.820007,20328.896237,10681.642782


## Test inflator

### Does it do anything?

In [24]:
tmp = pm.list_transformers[5]
columns_lt = X_train.columns.intersection(other=tmp.list_cols)

print('Before:')
display(X_train.groupby(by=X_train[tmp.str_datecol].dt.year)[columns_lt].max().T)

X_train = tmp.transform(X=X_train)

print(pd.Series(data=tmp.dict_inflation_rate))
print('After:')
display(X_train.groupby(by=X_train[tmp.str_datecol].dt.year)[columns_lt].max().T)

Before:


applicationdate__app,2013,2014,2015,2016,2017
linkf060__tu,771.0,605.00,860.0,820.0,1775.0
linkf045__tu,870.0,1838.00,3830.0,2299.0,3587.0
linkf079__tu,2337.0,6650.00,7497.0,52530.0,9719.0
linkf076__tu,870.0,1838.00,3830.0,2299.0,3587.0
linkf132__tu,16000.0,18000.00,122850.0,98000.0,26000.0
...,...,...,...,...,...
lienjudgmentdollartotal__ln,101477.0,385968.00,414011.0,450000.0,588472.0
assetpropnewestsaleprice__ln,508000.0,892000.00,1170106.0,725915.0,998280.0
fltinsureddisabilityamount__app,0.0,1219.74,0.0,0.0,0.0
fltdowncash__app,3000.0,10000.00,5000.0,8000.0,3500.0


100%|██████████| 263/263 [00:00<00:00, 2968.49it/s]

Inflate to 2021 dollars (non-automobile): 0.16844 sec.
2013    1.163176
2014    1.144608
2015    1.143251
2016    1.129009
2017    1.105459
2018    1.079102
2019    1.059897
dtype: float64
After:


applicationdate__app,2013,2014,2015,2016,2017
linkf060__tu,896.808724,6.924880e+02,9.831961e+02,925.787165,1.962189e+03
linkf045__tu,1011.963152,2.103790e+03,4.378653e+03,2595.591087,3.965280e+03
linkf079__tu,2718.342398,7.611645e+03,8.570955e+03,59306.828967,1.074395e+04
linkf076__tu,1011.963152,2.103790e+03,4.378653e+03,2595.591087,3.965280e+03
linkf132__tu,18610.816588,2.060295e+04,1.404484e+05,110642.856250,2.874192e+04
...,...,...,...,...,...
lienjudgmentdollartotal__ln,118035.614684,4.417822e+05,4.733186e+05,508053.931760,6.505314e+05
assetpropnewestsaleprice__ln,590893.426684,1.020991e+06,1.337725e+06,819564.377497,1.103557e+06
fltinsureddisabilityamount__app,0.000000,1.396125e+03,0.000000e+00,0.000000,0.000000e+00
fltdowncash__app,3489.528110,1.144608e+04,5.716257e+03,9032.069898,3.869105e+03


## Test clip values

### Does it do anything?

In [25]:
tmp = pm.list_transformers[6]
columns_lt = X_train.columns.intersection(other=tmp.list_cols)

print('Before:', X_train[columns_lt].agg(func=['min', 'max']).T.sort_values(by=['min', 'max']), sep='\n')

X_train = tmp.transform(X=X_train)

print('After:', X_train[columns_lt].agg(func=['min', 'max']).T.sort_values(by=['min', 'max']), sep='\n')

Before:
                                         min            max
fltapproveddowntotal__app      -17627.932837   30814.657107
fltallowance__app              -10842.999913    4703.883893
fltapprovedpayment__app         -7056.015040   24443.889837
aggs909__tu                       -10.468584   28696.751583
aggs906__tu                       -10.468584   34999.270855
...                                      ...            ...
linkf079__tu                        0.000000   59306.828967
linkf199__tu                        0.000000   67016.535407
linkf132__tu                        0.000000  140448.425640
fltapprovedpricewholesale__app      0.000000  294295.528068
fltgrossmonthly__income_sum         0.000000  816593.349638

[278 rows x 2 columns]


100%|██████████| 278/278 [00:00<00:00, 1018.50it/s]


Clip negative dollar values to zero (automobile and non-automobile): 0.35193 sec.
After:
              min           max
linka060__tu  0.0  2.583748e+02
linka061__tu  0.0  2.583748e+02
linka063__tu  0.0  4.014714e+02
linka064__tu  0.0  4.014714e+02
linka062__tu  0.0  4.014714e+02
...           ...           ...
bkc322__tu    0.0  1.163176e+09
rev321__tu    0.0  1.163176e+09
rev322__tu    0.0  1.163176e+09
rev323__tu    0.0  1.163176e+09
rev324__tu    0.0  1.163176e+09

[278 rows x 2 columns]


## Test clip values

### Does it do anything?

In [26]:
tmp = pm.list_transformers[7]
columns_lt = X_train.columns.intersection(other=tmp.list_cols)

print('After:', X_train[columns_lt].agg(func=['min', 'max']).T.sort_values(by=['min', 'max']), sep='\n')

X_train = tmp.transform(X=X_train)

print('After:', X_train[columns_lt].agg(func=['min', 'max']).T.sort_values(by=['min', 'max']), sep='\n')

After:
                               min  max
fltgrossmonthly__income_count  1.0  5.0


100%|██████████| 1/1 [00:00<00:00, 459.55it/s]

Clip number of income sources to 2: 0.0075692 sec.
After:
                               min  max
fltgrossmonthly__income_count  1.0  2.0


## Test custom imputer

### Does it do anything?

In [27]:
tmp = pm.list_transformers[8]
imputation_dt = {
    column_sr: value_ft 
    for column_sr, value_ft in tmp.dict_imputation.items() 
    if column_sr in X_train.columns}

print('Before:')
for column_sr, value_ft in imputation_dt.items():
    print(X_train[column_sr].value_counts().head(), end='\n' + '-' * int(8e1))

X_train = tmp.transform(X=X_train)

print('=' * int(8e1), 'After:', sep='\n')
for column_sr, value_ft in imputation_dt.items():
    print(X_train[column_sr].value_counts().head(), end='\n' + '-' * int(8e1))

Before:


0it [00:00, ?it/s]

Custom imputer: 0.0060001 sec.
After:


### Would it do anything?

In [28]:
imputation_dt = tmp.dict_imputation
test_df = pd.DataFrame(data={key_sr: np.nan for key_sr in imputation_dt.keys()}, index=[0])

print('Before:')
for column_sr, value_ft in imputation_dt.items():
    print(test_df[column_sr].value_counts().head(), end='\n' + '-' * int(8e1))

test_df = tmp.transform(X=test_df)

print('=' * int(8e1), 'After:', sep='\n')
for column_sr, value_ft in imputation_dt.items():
    print(test_df[column_sr].value_counts().head(), end='\n' + '-' * int(8e1))

Before:
Series([], Name: count, dtype: int64)
--------------------------------------------------------------------------------

100%|██████████| 1/1 [00:00<00:00, 873.63it/s]

Custom imputer: 0.0060834 sec.
After:
intservicecontractmileage__app
125000.0    1
Name: count, dtype: int64
--------------------------------------------------------------------------------

## Test imputer

### Does it do anything?

In [29]:
tmp = pm.list_transformers[9]

print(f'Before: Zeros: {(X_train == 0).sum().sum():,} | Missing values: {X_train.isna().sum().sum():,}')

X_train = tmp.transform(X=X_train)

print(f'After: Zeros: {(X_train == 0).sum().sum():,} | Missing values: {X_train.isna().sum().sum():,}')

Before: Zeros: 5,919,056 | Missing values: 1,736,377
Imputer: 0.3513 sec.
After: Zeros: 7,655,433 | Missing values: 0


## Test feature value replacer

### Does it do anything?

In [30]:
tmp = pm.list_transformers[10]
to_replace_dt = {
    key_sr: list(value_dt.keys()) + list(value_dt.values())
    for key_sr, value_dt in tmp.dict_value_replace.items() 
    if key_sr in X_train.columns}

print('Before:')
for column_sr, values_lt in to_replace_dt.items():
    value_counts_ss = X_train[column_sr].value_counts()
    print(value_counts_ss[value_counts_ss.index.isin(values=values_lt)], end='\n' + '-' * int(8e1))

X_train = tmp.transform(X=X_train)

print('=' * int(8e1), 'After:', sep='\n')
for column_sr, values_lt in to_replace_dt.items():
    value_counts_ss = X_train[column_sr].value_counts()
    print(value_counts_ss[value_counts_ss.index.isin(values=values_lt)], end='\n' + '-' * int(8e1))

Before:
fltamountfinanced__app
0.0    11245
Name: count, dtype: int64
--------------------------------------------------------------------------------fltapprovedpricewholesale__app
0.0    5306
Name: count, dtype: int64
--------------------------------------------------------------------------------

100%|██████████| 2/2 [00:00<00:00, 295.17it/s]

Replace zeros with predetermined value and change ram to dodge: 0.017967 sec.
After:
fltamountfinanced__app
45000.0    11245
Name: count, dtype: int64
--------------------------------------------------------------------------------fltapprovedpricewholesale__app
28125.0    5306
Name: count, dtype: int64
--------------------------------------------------------------------------------

## Test date features

### Does it do anything?

In [31]:
tmp = pm.list_transformers[11]

print('Before:')
display(X_train.filter(like='applicationdate__app').head())

X_train = tmp.transform(X=X_train)

print('After:')
display(X_train.filter(like='applicationdate__app').head())

Before:


,applicationdate__app
613648,2015-10-21
54406,2013-12-20
1241911,2017-09-23
738908,2016-03-05
567065,2015-08-28


Date features: 0.022992 sec.
After:


/home/ec2-user/SageMaker/20231010_gen_xii/ad_hoc/check_preprocessing/preprocessing.py:341: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['ENG-applicationdate__app_month'] = X['applicationdate__app'].dt.month
/home/ec2-user/SageMaker/20231010_gen_xii/ad_hoc/check_preprocessing/preprocessing.py:342: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['ENG-applicationdate__app_quarter'] = X['applicationdate__app'].dt.quarter


,applicationdate__app,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter
613648,2015-10-21,10,4
54406,2013-12-20,12,4
1241911,2017-09-23,9,3
738908,2016-03-05,3,1
567065,2015-08-28,8,3


## Test round binning

### Does it do anything?

In [32]:
tmp = pm.list_transformers[12]
columns_lt = list(tmp.dict_round.keys())

print('Before:')
display(X_train[columns_lt].head())

X_train = tmp.transform(X=X_train)

print('After:')
display(X_train[columns_lt].head())

Before:


,fltamountfinanced__app,fltapprovedpricewholesale__app,fltgrossmonthly__income_sum
613648,27942.56978,28125.000000,3951.076589
54406,45000.00000,24530.301642,7560.644239
1241911,45000.00000,11839.544439,2210.917102
738908,45000.00000,28125.000000,2540.269659
567065,45000.00000,20056.435973,2012.122337


100%|██████████| 3/3 [00:00<00:00, 906.88it/s]

Round income (for PTI), amount financed and vehicle values for (LTV): 0.0080519 sec.
After:


,fltamountfinanced__app,fltapprovedpricewholesale__app,fltgrossmonthly__income_sum
613648,28000.0,28000.0,4000.0
54406,45000.0,24500.0,7500.0
1241911,45000.0,12000.0,2000.0
738908,45000.0,28000.0,2500.0
567065,45000.0,20000.0,2000.0


## Test feature engineering

### Does it do anything?

In [33]:
tmp = pm.list_transformers[13]

print('Before:')
display(X_train.filter(like='ENG'))

X_train = tmp.transform(X=X_train)

print('After:')
display(X_train.filter(like='ENG'))

Before:


,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter
613648,10,4
54406,12,4
1241911,9,3
738908,3,1
567065,8,3
...,...,...
780225,4,2
1036445,1,1
13993,10,4
359605,1,1


Engineer PTI and LTV: 0.0058929 sec.
After:


/home/ec2-user/SageMaker/20231010_gen_xii/ad_hoc/check_preprocessing/preprocessing.py:404: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['ENG-payment_to_income'] = X['fltapprovedpayment__app'] / X['fltgrossmonthly__income_sum']
/home/ec2-user/SageMaker/20231010_gen_xii/ad_hoc/check_preprocessing/preprocessing.py:409: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['ENG-loan_to_value'] = X['fltamountfinanced__app'] / X['fltapprovedpricewholesale__app']


,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value
613648,10,4,0.155679,1.000000
54406,12,4,0.000000,1.836735
1241911,9,3,0.000000,3.750000
738908,3,1,0.000000,1.607143
567065,8,3,0.258518,2.250000
...,...,...,...,...
780225,4,2,0.190797,2.812500
1036445,1,1,0.094737,1.607143
13993,10,4,0.000000,1.607143
359605,1,1,0.135062,2.500000


## Test replace inf

### Does it do anything?

In [34]:
tmp = pm.list_transformers[14]

print(f'Before: {X_train.map(func=lambda x: x in [np.inf, -np.inf]).sum().sum():,}')

X_train = tmp.transform(X=X_train)

print(f'After: {X_train.map(func=lambda x: x in [np.inf, -np.inf]).sum().sum():,}')

Before: 280


100%|██████████| 1569/1569 [00:01<00:00, 1495.38it/s]


Replace inf and -inf with NaN: 1.3201 sec.
After: 0


## Test imputer

### Does it do anything?

In [35]:
tmp = pm.list_transformers[15]

print(f'Before: Zeros: {(X_train == 0).sum().sum():,} | Missing values: {X_train.isna().sum().sum():,}')

X_train = tmp.transform(X=X_train)

print(f'After: Zeros: {(X_train == 0).sum().sum():,} | Missing values: {X_train.isna().sum().sum():,}')

Before: Zeros: 7,643,451 | Missing values: 420
Imputer: 0.19578 sec.
After: Zeros: 7,643,871 | Missing values: 0


## Test term mapper

### Does it do anything?

In [36]:
tmp = pm.list_transformers[16]
term_sr = 'intterm__app'

print(f'Before: {X_train[term_sr].value_counts()}')

X_train = tmp.transform(X=X_train)

print(f'After: {X_train[term_sr].value_counts()}')

Before: intterm__app
0.0     11245
72.0      978
66.0      218
60.0       99
54.0       40
48.0       14
36.0        6
69.0        1
68.0        1
70.0        1
24.0        1
Name: count, dtype: int64
Map term: 0.0053165 sec.
After: intterm__app
72    12444
60      139
48       14
36        6
24        1
Name: count, dtype: int64


## Would it do anything?

In [37]:
test_df = pd.DataFrame(data={term_sr: [None, -1, 0] + list(range(6, 80, 6))})

test_df.assign(new_term = lambda x: x[term_sr].apply(func=preprocessing.custom_mapping_term))

,intterm__app,new_term
0,NaN,72
1,-1.0,12
2,0.0,72
3,6.0,12
4,12.0,12
5,18.0,24
6,24.0,24
7,30.0,36
8,36.0,36
9,42.0,48


## Test PTI mapper

### Does it do anything?

In [38]:
tmp = pm.list_transformers[17]
pti_sr = 'ENG-payment_to_income'

print('Before:')
with pd.option_context('display.max_rows', None):
    display(X_train[pti_sr].round(decimals=2).value_counts().sort_index())

X_train = tmp.transform(X=X_train)

print('After:')
with pd.option_context('display.max_rows', None):
    display(X_train[pti_sr].round(decimals=2).value_counts().sort_index())

Before:


ENG-payment_to_income
0.00    4960
0.01      11
0.02      24
0.03      38
0.04     106
0.05     140
0.06     271
0.07     360
0.08     443
0.09     596
0.10     587
0.11     664
0.12     583
0.13     629
0.14     459
0.15     767
0.16     401
0.17     350
0.18      95
0.19     111
0.20     125
0.21     101
0.22      44
0.23      94
0.24      42
0.25      39
0.26      95
0.27      36
0.28      29
0.29      29
0.30      18
0.31      29
0.32      22
0.33       8
0.34      45
0.35      16
0.36       4
0.37      14
0.38       6
0.39      10
0.40      10
0.41       6
0.42       1
0.43       5
0.44       9
0.45       5
0.46      19
0.47       7
0.48       9
0.49       7
0.50       7
0.51      31
0.52       7
0.53       2
0.54       3
0.55       4
0.57       2
0.58       2
0.59       2
0.60       1
0.62       3
0.63       1
0.64       2
0.65       2
0.66       1
0.67       2
0.68       2
0.69       1
0.70       2
0.71       1
0.72       1
0.74       1
0.77       1
0.79       3
0.80       2
0.8

Map PTI: 0.0062942 sec.
After:


ENG-payment_to_income
0.00      58
0.03     393
0.06    1154
0.09    1952
0.12    1789
0.15    7258
Name: count, dtype: int64

## Would it do anything?

In [39]:
test_df = pd.DataFrame(data={pti_sr: [None] + np.arange(start=-1e-2, stop=2e-1, step=1e-2).tolist()})

test_df.assign(new_term = lambda x: x[pti_sr].apply(func=preprocessing.custom_mapping_pti))

,ENG-payment_to_income,new_term
0,NaN,0.15
1,-0.01,0.15
2,0.00,0.15
3,0.01,0.00
4,0.02,0.00
5,0.03,0.00
6,0.04,0.03
7,0.05,0.03
8,0.06,0.06
9,0.07,0.06


## Test round binning

### Does it do anything?

In [40]:
tmp = pm.list_transformers[18]
columns_lt = X_train.columns.intersection(other=list(tmp.dict_round.keys())).tolist()

print('Before:')
display(X_train[columns_lt].head())

X_train = tmp.transform(X=X_train)

print('After:')
display(X_train[columns_lt].head())

Before:


,fltapprovedservicecontract__app,fltgapinsurance__app,fltapprovedpricewholesale__app,fltapproveddowntotal__app,fltamountfinanced__app,fltdowncash__app
613648,0.0,676.123479,28000.0,0.000000,28000.0,0.0
54406,0.0,0.000000,24500.0,1744.764055,45000.0,0.0
1241911,0.0,0.000000,12000.0,0.000000,45000.0,0.0
738908,0.0,0.000000,28000.0,0.000000,45000.0,0.0
567065,0.0,0.000000,20000.0,1714.876992,45000.0,0.0


100%|██████████| 6/6 [00:00<00:00, 800.87it/s]

Round values: 0.012243 sec.
After:


,fltapprovedservicecontract__app,fltgapinsurance__app,fltapprovedpricewholesale__app,fltapproveddowntotal__app,fltamountfinanced__app,fltdowncash__app
613648,0.0,500.0,28000.0,0.0,28000.0,0.0
54406,0.0,0.0,24500.0,1500.0,45000.0,0.0
1241911,0.0,0.0,12000.0,0.0,45000.0,0.0
738908,0.0,0.0,28000.0,0.0,45000.0,0.0
567065,0.0,0.0,20000.0,1500.0,45000.0,0.0
